# Lab 03-05 · Estimation and Hypothesis Testing for Engineers

Companion to chapter [03-05](../../chapters/03-05-estimation-and-hypothesis-testing.md). Handout: [README.md](README.md).

Statistics makes promises about what happens over many repetitions of an experiment: "95% of intervals cover the truth", "5% false alarms". Here you repeat the experiments thousands of times and check the promises. Cells marked **🔮 Predict** ask for a prediction *before* running. The seed is fixed.

In [1]:
import math, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass
rng = np.random.default_rng(2026)

## Part 1 — Why divide by $n - 1$

**🔮 Predict:** with samples of $n = 5$ from a distribution with σ² = 1, what does dividing the sum of squared deviations by $n$ give on average? By $n - 1$?

In [2]:
n, reps = 5, 200_000
x = rng.standard_normal((reps, n))
print(f"divide by n    : mean {x.var(axis=1, ddof=0).mean():.4f}   (theory (n-1)/n = {(n - 1) / n:.4f})")
print(f"divide by n - 1: mean {x.var(axis=1, ddof=1).mean():.4f}   (theory 1)")
print(f"sample std s   : mean {x.std(axis=1, ddof=1).mean():.4f}   (slightly below 1: the square root is concave)")

divide by n    : mean 0.8007   (theory (n-1)/n = 0.8000)
divide by n - 1: mean 1.0008   (theory 1)
sample std s   : mean 0.9403   (slightly below 1: the square root is concave)


## Part 2 — Do 95% intervals cover 95% of the time?

Repeat a 5-sample and a 30-sample experiment 20,000 times, build a "z-interval" (1.96 with the sample $s$) and a t-interval each time, and count how often each covers the true mean.

**🔮 Predict:** the coverage of the z-interval at $n = 5$.

In [3]:
for n in (5, 30):
    x = rng.normal(1.1, 0.25, (20_000, n))
    m, s = x.mean(axis=1), x.std(axis=1, ddof=1)
    se = s / math.sqrt(n)
    tq = stats.t.ppf(0.975, n - 1)
    cov_z = np.mean(np.abs(m - 1.1) <= 1.96 * se)
    cov_t = np.mean(np.abs(m - 1.1) <= tq * se)
    print(f"n = {n:2d}: z-with-s coverage {cov_z:.4f} (theory {2 * stats.t.cdf(1.96, n - 1) - 1:.4f}),  t coverage {cov_t:.4f}")

n =  5: z-with-s coverage 0.8842 (theory 0.8784),  t coverage 0.9509
n = 30: z-with-s coverage 0.9379 (theory 0.9403),  t coverage 0.9472


## Part 3 — The chi-square interval for noise power

Each experiment records 50 samples of zero-mean noise with σ² = 1 and forms the 95% interval of Example 03-05.3.

In [4]:
nu = 50
lo_q, hi_q = stats.chi2.ppf([0.025, 0.975], nu)
ms = (rng.standard_normal((20_000, nu))**2).mean(axis=1)
covered = np.mean((nu * ms / hi_q <= 1) & (1 <= nu * ms / lo_q))
print(f"coverage {covered:.4f}; typical interval width {np.median(nu * ms / lo_q - nu * ms / hi_q):.3f} (in units of the true power)")

coverage 0.9486; typical interval width 0.831 (in units of the true power)


## Part 4 — The power of a counting test

Example 03-05.4 rejects "the vendor's rate is right" when a year's fleet count is 83 or less. Simulate many fleet-years at several true rates.

**🔮 Predict:** the power when the true rate is 20% below the vendor's.

In [5]:
lam0, kc = 1e4 * 1140e-9 * 8766, 83
for f in (1.0, 0.95, 0.9, 0.8, 0.7):
    counts = rng.poisson(f * lam0, 50_000)
    print(f"true rate {f:4.2f} x vendor: rejection rate {np.mean(counts <= kc):.4f}   (exact {stats.poisson.cdf(kc, f * lam0):.4f})")

true rate 1.00 x vendor: rejection rate 0.0472   (exact 0.0470)
true rate 0.95 x vendor: rejection rate 0.1202   (exact 0.1188)
true rate 0.90 x vendor: rejection rate 0.2544   (exact 0.2516)
true rate 0.80 x vendor: rejection rate 0.6597   (exact 0.6602)
true rate 0.70 x vendor: rejection rate 0.9452   (exact 0.9442)


At the vendor's own rate the test rejects about 4.7% of the time: the significance level. A 10% reduction is detected only about a quarter of the time, a 20% reduction about two-thirds of the time.

## Part 5 — How uncertain is a measured Cpk?

The process of Example 03-05.5 has true $C_{pk} = 1.389$. Run many 30-part and 125-part capability studies.

In [6]:
mu, sigma, lsl, usl = 1.205, 0.006, 1.170, 1.230
true = min(usl - mu, mu - lsl) / (3 * sigma)
for n in (30, 125):
    x = rng.normal(mu, sigma, (20_000, n))
    m, s = x.mean(axis=1), x.std(axis=1, ddof=1)
    cpk = np.minimum(usl - m, m - lsl) / (3 * s)
    se_formula = math.sqrt(1 / (9 * n) + true**2 / (2 * (n - 1)))
    print(f"n = {n:3d}: mean Cpk-hat {cpk.mean():.3f}, sd {cpk.std():.4f} (formula {se_formula:.4f}), "
          f"P(Cpk-hat < 1.33) = {np.mean(cpk < 1.33):.3f}")

n =  30: mean Cpk-hat 1.425, sd 0.2046 (formula 0.1923), P(Cpk-hat < 1.33) = 0.346
n = 125: mean Cpk-hat 1.397, sd 0.0948 (formula 0.0931), P(Cpk-hat < 1.33) = 0.247


With 30 parts a process whose true Cpk is 1.389 "fails" 1.33 about 35% of the time; with 125 parts, still about 25%. Note also that the 30-part estimate is biased slightly high on average. Real Cpk near a threshold is hard to demonstrate in either direction.

## Part 6 — Weibull plots from small tests

Draw many eight-unit stress tests from a Weibull with γ = 2.5 and α = 1189 h (close to Example 03-05.6) and fit each on a Weibull plot.

**🔮 Predict:** how much does the fitted shape scatter between eight-unit tests?

In [7]:
def weibull_fit(t):
    t = np.sort(t); n = t.size
    F = (np.arange(1, n + 1) - 0.3) / (n + 0.4)
    slope, icpt = np.polyfit(np.log(t), np.log(-np.log(1 - F)), 1)
    return slope, math.exp(-icpt / slope)
fits = np.array([weibull_fit(1189 * rng.weibull(2.5, 8)) for _ in range(5000)])
g = fits[:, 0]
print(f"shape estimate: median {np.median(g):.2f}, 5th to 95th percentile {np.percentile(g, 5):.2f} to {np.percentile(g, 95):.2f}")
print(f"relative spread (sd/median): {g.std() / np.median(g):.2f}")
print(f"Example 03-05.6 data fit: {weibull_fit(np.array([457, 714, 823, 912, 976, 1182, 1600, 1706.]))}")

shape estimate: median 2.27, 5th to 95th percentile 1.30 to 4.15
relative spread (sd/median): 0.40
Example 03-05.6 data fit: (np.float64(2.4982274429347417), 1189.2217866232277)


Eight failures pin the shape down only to about ±40% (one standard deviation), with a 90% range of roughly 1.3 to 4.2 when the truth is 2.5, and the least-squares estimate is biased low. Infant mortality (γ < 1) is still distinguishable from wear-out, but γ = 2 and γ = 3 are not. Reliability engineers therefore test larger samples, pool data across stress cells, and use maximum-likelihood fits with confidence bounds ([25](../../../25-fabrication-packaging-test-and-reliability/README.md)).

## Part 7 — Do FIT upper bounds cover the truth?

Simulate HTOL-style tests (231 units, 1000 h, acceleration 77.65) on a part whose true field rate is 150 FIT, and compute the 90% upper bound from each test's failure count.

In [8]:
true_fit, D = 150.0, 231 * 1000 * 77.65
lam = true_fit * 1e-9 * D
c = rng.poisson(lam, 100_000)
bound = stats.chi2.ppf(0.9, 2 * c + 2) / 2 / D * 1e9
print(f"expected failures per test {lam:.3f}; P(zero failures) {np.exp(-lam):.3f}")
print(f"90% bound covers the true 150 FIT in {np.mean(bound >= true_fit):.4f} of tests (at least 0.90 by construction)")

expected failures per test 2.691; P(zero failures) 0.068
90% bound covers the true 150 FIT in 0.9327 of tests (at least 0.90 by construction)


The bound misses only when a test happens to see zero failures (bound 128.4 FIT, below the true 150). Because failure counts are discrete, the actual coverage exceeds the nominal 90%: the bound is conservative.

## Your turn

1. In Part 2, replace the Gaussian data with exponential data (a skewed distribution) at $n = 5$. Does the t-interval still cover 95%?
2. In Part 6, repeat with 30-unit tests. How much does the scatter of γ̂ shrink, and does it follow $1/\sqrt n$?